# Project 5 — Advanced Solution

This notebook provides production-quality solutions for both project goals:

1. A **class-based context manager** that lazily reads CSV records as named tuples.
2. A **generator-based context manager** implemented with `contextlib.contextmanager`.

Both solutions:

- require only a file name/path;
- use `csv.reader`;
- detect the CSV dialect with `csv.Sniffer`;
- rewind the file after dialect detection;
- preserve every data value as a string;
- validate the header instead of silently changing field names;
- close the file reliably, including when exceptions occur;
- produce records lazily rather than loading the entire file into memory.


## Design decisions

- Files are opened with `newline=''`, as recommended by the `csv` module.
- `utf-8-sig` is used so a UTF-8 byte-order mark, when present, does not become part of the first field name.
- Only a bounded sample is read for dialect detection; the data rows remain lazy.
- Invalid, duplicated, blank, keyword, or underscore-prefixed headers raise a clear exception. This is safer than using `namedtuple(..., rename=True)`, which would silently change the CSV field names.
- Iterators are intended to be consumed **inside** their `with` block because the underlying file is closed when the context exits.


In [8]:
import csv
import keyword
import re
from collections import namedtuple
from contextlib import contextmanager
from itertools import islice
from pathlib import Path


_SNIFF_SAMPLE_SIZE = 8 * 1024


## Shared validation and parsing helpers

The two public solutions share these small helpers so behavior remains consistent and parsing logic is not duplicated.


In [9]:
class CSVNamedTupleError(Exception):
    """Base exception for this project."""


class CSVHeaderError(CSVNamedTupleError, ValueError):
    """Raised when a CSV header cannot be used as named-tuple fields."""


class CSVRowError(CSVNamedTupleError, ValueError):
    """Raised when a data row has an unexpected number of fields."""


_COMMON_DELIMITERS = ",;\t|:"


def _fallback_dialect(sample):
    """Build a conservative fallback dialect from the first non-empty line."""
    first_line = next(
        (line for line in sample.splitlines() if line.strip()),
        "",
    )

    # Prefer the delimiter occurring most often in the header.
    # Comma wins ties, which matches the normal CSV default.
    delimiter = max(
        _COMMON_DELIMITERS,
        key=lambda candidate: (first_line.count(candidate), candidate == ","),
    )

    if first_line.count(delimiter) == 0:
        delimiter = ","

    class FallbackDialect(csv.excel):
        pass

    FallbackDialect.delimiter = delimiter
    return FallbackDialect


def _sniff_dialect(file_obj):
    """Detect a usable CSV dialect from a bounded sample and rewind the file."""
    sample = file_obj.read(_SNIFF_SAMPLE_SIZE)
    file_obj.seek(0)

    if not sample:
        raise CSVHeaderError("The CSV file is empty; a header row is required.")

    try:
        dialect = csv.Sniffer().sniff(
            sample,
            delimiters=_COMMON_DELIMITERS,
        )
    except (csv.Error, TypeError, ValueError):
        dialect = _fallback_dialect(sample)

    delimiter = getattr(dialect, "delimiter", None)
    delimiter_is_valid = (
        isinstance(delimiter, str)
        and len(delimiter) == 1
        and delimiter not in "\r\n\x00"
    )

    if not delimiter_is_valid:
        dialect = _fallback_dialect(sample)

    # Validate the dialect at reader-construction time. Some Python versions
    # allow Sniffer to return a dialect that csv.reader later rejects.
    try:
        csv.reader([], dialect=dialect)
    except (csv.Error, TypeError, ValueError):
        dialect = _fallback_dialect(sample)
        csv.reader([], dialect=dialect)

    return dialect


def _validate_header(header, path):
    """Validate that header values are legal, unique named-tuple fields."""
    if not header:
        raise CSVHeaderError(
            "CSV file {!s} has an empty header row.".format(path)
        )

    errors = []

    blank_fields = [index + 1 for index, name in enumerate(header) if not name]
    if blank_fields:
        errors.append("blank field name(s) at position(s) {}".format(blank_fields))

    invalid_fields = [
        name
        for name in header
        if name
        and (
            not name.isidentifier()
            or keyword.iskeyword(name)
            or name.startswith("_")
        )
    ]
    if invalid_fields:
        errors.append(
            "invalid Python field name(s): {}".format(invalid_fields)
        )

    duplicates = sorted({name for name in header if header.count(name) > 1})
    if duplicates:
        errors.append("duplicate field name(s): {}".format(duplicates))

    if errors:
        raise CSVHeaderError(
            "Invalid header in {!s}: {}. ".format(path, "; ".join(errors))
            + "CSV headers must be unique valid Python identifiers because "
            + "they become named-tuple field names."
        )


def _row_type_name(path):
    """Create a valid, descriptive named-tuple type name from a file path."""
    stem = re.sub(r"\W+", "_", path.stem).strip("_")
    if not stem or stem[0].isdigit() or keyword.iskeyword(stem):
        stem = "CSV_{}".format(stem or "Data")
    return "{}Row".format(stem)


def _prepare_reader(file_obj, path):
    """Create the csv.reader, consume its header, and build the row type."""
    dialect = _sniff_dialect(file_obj)
    reader = csv.reader(file_obj, dialect=dialect)

    try:
        header = next(reader)
    except StopIteration:
        raise CSVHeaderError(
            "CSV file {!s} does not contain a header row.".format(path)
        )

    _validate_header(header, path)
    row_type = namedtuple(_row_type_name(path), header)
    return reader, row_type, tuple(header)


def _build_record(raw_row, row_type, fieldnames, line_number, path):
    """Validate one raw row and convert it to the generated named tuple."""
    expected = len(fieldnames)
    actual = len(raw_row)

    if actual != expected:
        raise CSVRowError(
            "Malformed row in {!s} near CSV line {}: expected {} fields, "
            "received {}. Row={!r}".format(
                path, line_number, expected, actual, raw_row
            )
        )

    return row_type(*raw_row)

# Goal 1 — Class-based context manager

`CSVNamedTupleReader` implements both the context-manager protocol and iterator protocol:

- `__enter__` opens and initializes the CSV reader.
- `__exit__` always closes the file.
- `__iter__` and `__next__` make the object a lazy one-pass iterator.
- `fieldnames` exposes the validated header while inside the context.


In [10]:
class CSVNamedTupleReader:
    """Lazily read CSV rows as named tuples inside a managed context."""

    def __init__(self, file_name):
        self._path = Path(file_name)
        self._file = None
        self._reader = None
        self._row_type = None
        self._fieldnames = None

    def __enter__(self):
        if self._file is not None:
            raise RuntimeError("This CSV reader is already inside a context.")

        self._file = self._path.open(
            mode="r",
            encoding="utf-8-sig",
            newline="",
        )

        try:
            (
                self._reader,
                self._row_type,
                self._fieldnames,
            ) = _prepare_reader(self._file, self._path)
        except Exception:
            self._file.close()
            self._file = None
            raise

        return self

    def __exit__(self, exc_type, exc_value, traceback):
        file_obj = self._file

        # Clear internal references even if close() itself were to fail.
        self._file = None
        self._reader = None
        self._row_type = None
        self._fieldnames = None

        if file_obj is not None:
            file_obj.close()

        # Returning False propagates exceptions raised inside the with block.
        return False

    def __iter__(self):
        return self

    def __next__(self):
        if self._reader is None:
            raise RuntimeError(
                "CSVNamedTupleReader must be consumed inside a with block."
            )

        raw_row = next(self._reader)
        return _build_record(
            raw_row=raw_row,
            row_type=self._row_type,
            fieldnames=self._fieldnames,
            line_number=self._reader.line_num,
            path=self._path,
        )

    @property
    def fieldnames(self):
        if self._fieldnames is None:
            raise RuntimeError("fieldnames is available only inside the context.")
        return self._fieldnames

    @property
    def row_type(self):
        if self._row_type is None:
            raise RuntimeError("row_type is available only inside the context.")
        return self._row_type

    @property
    def is_open(self):
        return self._file is not None and not self._file.closed


## Goal 1 usage

`islice` consumes only the requested rows, demonstrating lazy iteration. The examples are guarded so the notebook still runs when the project CSV files are not in the current directory.


In [11]:
cars_path = Path("cars.csv")

if cars_path.exists():
    with CSVNamedTupleReader(cars_path) as cars:
        print("Fields:", cars.fieldnames)
        print("Generated type:", cars.row_type)
        print()

        for car in islice(cars, 5):
            print(car)
            print("Car name:", car.Car)
            print("Origin:", car.Origin)
            print()
else:
    print("cars.csv was not found in", Path.cwd())


Fields: ('Car', 'MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight', 'Acceleration', 'Model', 'Origin')
Generated type: <class '__main__.carsRow'>

carsRow(Car='Chevrolet Chevelle Malibu', MPG='18.0', Cylinders='8', Displacement='307.0', Horsepower='130.0', Weight='3504.', Acceleration='12.0', Model='70', Origin='US')
Car name: Chevrolet Chevelle Malibu
Origin: US

carsRow(Car='Buick Skylark 320', MPG='15.0', Cylinders='8', Displacement='350.0', Horsepower='165.0', Weight='3693.', Acceleration='11.5', Model='70', Origin='US')
Car name: Buick Skylark 320
Origin: US

carsRow(Car='Plymouth Satellite', MPG='18.0', Cylinders='8', Displacement='318.0', Horsepower='150.0', Weight='3436.', Acceleration='11.0', Model='70', Origin='US')
Car name: Plymouth Satellite
Origin: US

carsRow(Car='AMC Rebel SST', MPG='16.0', Cylinders='8', Displacement='304.0', Horsepower='150.0', Weight='3433.', Acceleration='12.0', Model='70', Origin='US')
Car name: AMC Rebel SST
Origin: US

carsRow(Car='Ford T

# Goal 2 — Generator function with `contextmanager`

The decorated function manages the file resource, while the nested generator lazily converts each raw CSV row into the generated named-tuple type.


In [12]:
@contextmanager
def csv_namedtuple_reader(file_name):
    """Yield a lazy iterator of named tuples for the supplied CSV file."""
    path = Path(file_name)
    file_obj = path.open(
        mode="r",
        encoding="utf-8-sig",
        newline="",
    )

    try:
        reader, row_type, fieldnames = _prepare_reader(file_obj, path)

        def records():
            for raw_row in reader:
                yield _build_record(
                    raw_row=raw_row,
                    row_type=row_type,
                    fieldnames=fieldnames,
                    line_number=reader.line_num,
                    path=path,
                )

        yield records()
    finally:
        file_obj.close()


## Goal 2 usage


In [13]:
personal_info_path = Path("personal_info.csv")

if personal_info_path.exists():
    with csv_namedtuple_reader(personal_info_path) as people:
        for person in islice(people, 5):
            print(person)
            print("Name: {} {}".format(person.first_name, person.last_name))
            print("Language:", person.language)
            print()
else:
    print("personal_info.csv was not found in", Path.cwd())


personal_infoRow(ssn='100-53-9824', first_name='Sebastiano', last_name='Tester', gender='Male', language='Icelandic')
Name: Sebastiano Tester
Language: Icelandic

personal_infoRow(ssn='101-71-4702', first_name='Cayla', last_name='MacDonagh', gender='Female', language='Lao')
Name: Cayla MacDonagh
Language: Lao

personal_infoRow(ssn='101-84-0356', first_name='Nomi', last_name='Lipprose', gender='Female', language='Yiddish')
Name: Nomi Lipprose
Language: Yiddish

personal_infoRow(ssn='104-22-0928', first_name='Justinian', last_name='Kunzelmann', gender='Male', language='Dhivehi')
Name: Justinian Kunzelmann
Language: Dhivehi

personal_infoRow(ssn='104-84-7144', first_name='Claudianus', last_name='Brixey', gender='Male', language='Afrikaans')
Name: Claudianus Brixey
Language: Afrikaans



# Automated tests

These tests use temporary CSV files, so they do not depend on `cars.csv` or `personal_info.csv`. They verify delimiter detection, named-tuple fields, string values, laziness-compatible iteration, cleanup, malformed rows, and invalid headers.


In [14]:
from tempfile import TemporaryDirectory


def _write_text(path, text):
    path.write_text(text, encoding="utf-8")


with TemporaryDirectory() as temp_dir_name:
    temp_dir = Path(temp_dir_name)

    semicolon_file = temp_dir / "cars_sample.csv"
    _write_text(
        semicolon_file,
        "Car;MPG;Origin\n"
        "Chevrolet Chevelle Malibu;18.0;US\n"
        "Toyota Corolla;31.0;Japan\n",
    )

    comma_file = temp_dir / "people_sample.csv"
    _write_text(
        comma_file,
        "ssn,first_name,last_name,language\n"
        "100-53-9824,Sebastiano,Tester,Icelandic\n"
        "101-71-4702,Cayla,MacDonagh,Lao\n",
    )

    # Goal 1: class-based manager.
    class_reader = CSVNamedTupleReader(semicolon_file)
    assert not class_reader.is_open

    with class_reader as rows:
        assert rows.is_open
        assert iter(rows) is rows
        assert rows.fieldnames == ("Car", "MPG", "Origin")

        first = next(rows)
        second = next(rows)

        assert type(first)._fields == ("Car", "MPG", "Origin")
        assert first.Car == "Chevrolet Chevelle Malibu"
        assert first.MPG == "18.0"
        assert first.Origin == "US"
        assert all(isinstance(value, str) for value in first)
        assert second.Car == "Toyota Corolla"

        try:
            next(rows)
        except StopIteration:
            pass
        else:
            raise AssertionError("Iterator should be exhausted.")

    assert not class_reader.is_open

    # Goal 2: generator-based manager.
    with csv_namedtuple_reader(comma_file) as rows:
        first_two = list(islice(rows, 2))

    assert len(first_two) == 2
    assert type(first_two[0])._fields == (
        "ssn",
        "first_name",
        "last_name",
        "language",
    )
    assert first_two[0].first_name == "Sebastiano"
    assert first_two[1].language == "Lao"
    assert all(
        isinstance(value, str)
        for record in first_two
        for value in record
    )

    # A malformed row must fail with a useful error.
    malformed_file = temp_dir / "malformed.csv"
    _write_text(malformed_file, "a,b,c\n1,2\n")

    try:
        with CSVNamedTupleReader(malformed_file) as rows:
            next(rows)
    except CSVRowError as exc:
        assert "expected 3 fields" in str(exc)
    else:
        raise AssertionError("Malformed rows should raise CSVRowError.")

    # Invalid named-tuple headers must not be silently renamed.
    invalid_header_file = temp_dir / "invalid_header.csv"
    _write_text(invalid_header_file, "first name,class,value\nA,B,C\n")

    try:
        with csv_namedtuple_reader(invalid_header_file) as rows:
            next(rows)
    except CSVHeaderError as exc:
        assert "invalid Python field name" in str(exc)
    else:
        raise AssertionError("Invalid headers should raise CSVHeaderError.")

print("All automated tests passed.")


All automated tests passed.


# Complexity and behavior

Let `n` be the number of data rows and `w` the maximum width of a row.

- **Time:** `O(n × w)` to consume the entire file.
- **Additional memory:** `O(w)` per currently materialized row, excluding the small bounded dialect sample.
- **Laziness:** neither implementation constructs a list of all records; each named tuple is created only when requested by iteration.
- **Resource safety:** files are closed by `__exit__` or `finally`, even when parsing or consumer code raises an exception.

Typical use:

```python
with CSVNamedTupleReader("cars.csv") as rows:
    first_car = next(rows)

with csv_namedtuple_reader("personal_info.csv") as rows:
    first_person = next(rows)
```
